# TKCE — honest-benchmark experiments (leak fixed)

**What changed:** `eye_movements` has a documented data leak (TabReD, ICLR 2025: grouped data + ID columns `lineNo/assgNo/titleNo/wordNo`; ID columns ALONE score 0.78 AUC under random splits). All runs here use the **de-leaked protocol**: ID columns dropped + split grouped by `assgNo`.

Three encodings compared on the fusion model (views `x` and `x+tree`, 3-member deep ensembles):
1. `infold` — naive encoding (forest saw the rows it encodes; label-leaky)
2. `oob` — **OOB-honest** encoding (train rows keep only bits from trees they were out-of-bag for)
3. `oob + tau 0.5` — OOB-honest **soft** bits (temperature-relaxed splits)

Plus a clean control dataset (**house_16H**, task 361063, no known leak).

### How to run
1. Runtime -> Change runtime type -> **GPU**.
2. **Run all.** (~40-60 min total)

### What to look for
- Honest tree ceiling on eye_movements is now ~0.55-0.61 (not 0.708!) — does `x+tree` **match/beat** it?
- `infold` vs `oob` difference = the **encoding-leakage measurement** (a key paper number)
- Train-AUC curves: `oob` should stop the instant snap to 1.0

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> set Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps  (OpenML is reachable again -> dataset downloads automatically)
!pip install -q openml catboost optuna

In [ ]:
# 4 · EXPERIMENT A — de-leaked eye_movements: infold vs OOB-honest vs OOB+soft
common = ('--task 361070 --drop-cols lineNo,assgNo,titleNo,wordNo --group-by assgNo '
          "--views 'x;x+tree' --ensemble 3 --epochs 400 --dropout 0.3 --l1 5e-5 "
          '--weight-decay 1e-3 --lr 3e-4 --device auto')
!python -u run_fusion.py {common} --encoding infold           --out results/fusion/em_infold
!python -u run_fusion.py {common} --encoding oob              --out results/fusion/em_oob
!python -u run_fusion.py {common} --encoding oob --tau 0.5    --out results/fusion/em_oob_soft

In [ ]:
# 5 · EXPERIMENT B — clean control dataset (house_16H, task 361063, no known leak)
commonB = ("--task 361063 --views 'x;x+tree' --ensemble 3 --epochs 400 --dropout 0.3 "
           '--l1 5e-5 --weight-decay 1e-3 --lr 3e-4 --device auto')
!python -u run_fusion.py {commonB} --encoding infold --out results/fusion/house_infold
!python -u run_fusion.py {commonB} --encoding oob    --out results/fusion/house_oob

In [ ]:
# 6 · show all figures
from IPython.display import Image, display
import glob, os
for d in ['em_infold','em_oob','em_oob_soft','house_infold','house_oob']:
    for f in sorted(glob.glob(f'results/fusion/{d}/*.png')):
        print(f'== {d} : {os.path.basename(f)} ==')
        display(Image(f))

In [ ]:
# 7 · download everything as one zip
import shutil
from google.colab import files
shutil.make_archive('fusion_honest_benchmark', 'zip', 'results/fusion')
files.download('fusion_honest_benchmark.zip')